# LeetCode 236 — Lowest Common Ancestor of a Binary Tree bằng Python

Link đề: https://leetcode.com/problems/lowest-common-ancestor-of-a-binary-tree/ · Độ khó: **Medium** · Chủ đề: Tree, Depth-First Search, Binary Tree, Lowest Common Ancestor

Skeleton LeetCode:

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, x):
#         self.val = x
#         self.left = None
#         self.right = None
class Solution:
    def lowestCommonAncestor(self, root: 'TreeNode', p: 'TreeNode', q: 'TreeNode') -> 'TreeNode':
        ...
```

Ghi chú: đề, ví dụ và constraints lấy từ đề gốc; mọi đáp án trong notebook đều được kiểm chứng bằng code và đối chiếu với một lời giải độc lập đi đúng theo **định nghĩa** (xét mọi nút, kiểm tra cả `p` và `q` nằm trong cây con của nút đó, lấy nút có cây con nhỏ nhất).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tìm **tổ tiên chung thấp nhất (LCA)** của hai nút `p`, `q` trong một cây **nhị phân bất kỳ** (không có tính chất sắp thứ tự như BST): nút **thấp nhất** có cả `p` và `q` là hậu duệ (một nút là hậu duệ của chính nó).
2. Biết dựng cây (`TreeNode`, `build_tree` từ level-order, sinh cây ngẫu nhiên) và lấy nút theo giá trị để dùng làm `p`, `q`.
3. Hiểu vì sao **không dùng được giá trị để chọn nhánh** (khác bài 235) và phải **tìm `p`, `q` bằng duyệt cây**.
4. Đi từ **kiểm tra mọi nút theo định nghĩa (`O(n²)`)** → **đường đi gốc → nút rồi tiền tố chung** → **bản đồ cha + tập tổ tiên** → **đệ quy hậu thứ tự (cách nộp bài)** → **bản đồ cha + độ sâu (như hai danh sách liên kết cắt nhau)**.
5. Tránh các lỗi kinh điển: dùng thuật toán BST cho cây thường, quên điều kiện cơ sở `root is p or root is q`, trả `left or right` mà bỏ qua trường hợp cả hai bên đều tìm thấy, và quên `path.pop()` khi quay lui.

## 2. Đề bài & ví dụ

Cho một cây nhị phân, hãy tìm **tổ tiên chung thấp nhất (LCA)** của hai nút cho trước trong cây.

Theo Wikipedia: "Tổ tiên chung thấp nhất giữa hai nút `p` và `q` là nút thấp nhất trong cây `T` có cả `p` và `q` là hậu duệ (ta cho phép một nút là hậu duệ của chính nó)."

**Ví dụ 1**

- Input: `root = [3,5,1,6,2,0,8,null,null,7,4]`, `p = 5`, `q = 1`
- Output: `3`
- Giải thích: LCA của nút `5` và nút `1` là `3`.

**Ví dụ 2**

- Input: `root = [3,5,1,6,2,0,8,null,null,7,4]`, `p = 5`, `q = 4`
- Output: `5`
- Giải thích: LCA của `5` và `4` là `5`, vì một nút có thể là hậu duệ của chính nó.

**Ví dụ 3**

- Input: `root = [1,2]`, `p = 1`, `q = 2`
- Output: `1`

**Ràng buộc:** số nút trong khoảng `[2, 10⁵]`; `-10⁹ <= Node.val <= 10⁹`; mọi `Node.val` **khác nhau**; `p != q`; `p` và `q` đều tồn tại trong cây.

In [1]:
from typing import List, Optional
import random
from collections import deque


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(values):
    """Dựng cây từ danh sách level-order (None = nút vắng mặt), giống định dạng của LeetCode."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i])
            queue.append(node.right)
        i += 1
    return root


def find_node(root, val):
    """Tìm nút có giá trị val để làm đầu vào p, q khi test."""
    stack = [root] if root else []
    while stack:
        node = stack.pop()
        if node.val == val:
            return node
        if node.left:
            stack.append(node.left)
        if node.right:
            stack.append(node.right)
    return None


examples = [
    ([3, 5, 1, 6, 2, 0, 8, None, None, 7, 4], 5, 1, 3),
    ([3, 5, 1, 6, 2, 0, 8, None, None, 7, 4], 5, 4, 5),
    ([1, 2], 1, 2, 1),
]
for values, pv, qv, expected in examples:
    root = build_tree(values)
    p, q = find_node(root, pv), find_node(root, qv)
    print(f"root={values}, p={p.val}, q={q.val}, kỳ vọng {expected}")

root=[3, 5, 1, 6, 2, 0, 8, None, None, 7, 4], p=5, q=1, kỳ vọng 3
root=[3, 5, 1, 6, 2, 0, 8, None, None, 7, 4], p=5, q=4, kỳ vọng 5
root=[1, 2], p=1, q=2, kỳ vọng 1


## 3. Kiến thức nền

### 3.1. Nút cây, `build_tree`, và `p`, `q` là nút (không phải số)

Đầu vào `p`, `q` là **đối tượng nút** trong cây, và kết quả cũng là một nút. Khi test ta dựng cây rồi dùng `find_node` để lấy đúng nút. Hai nút khác nhau có thể có cùng giá trị trong cây tổng quát, nhưng đề đảm bảo mọi giá trị khác nhau, nên so sánh bằng giá trị hay danh tính (`is`) đều được; ta dùng `is` cho chắc.

In [2]:
root = build_tree([3, 5, 1, 6, 2, 0, 8, None, None, 7, 4])
p = find_node(root, 5)
print(p.val, p.left.val, p.right.val, p.right.left.val, p.right.right.val)   # 5 6 2 7 4
print(find_node(root, 5) is p, find_node(root, 7) is p)                      # True False

5 6 2 7 4
True False


### 3.2. Vì sao không dùng được so sánh giá trị như trong BST

Ở BST, so sánh giá trị cho biết nên đi sang nhánh nào. Ở cây nhị phân thường, giá trị **không** có thứ tự: nút `5` lớn hơn gốc `3` nhưng lại nằm bên **trái**, còn `1` nhỏ hơn `3` thì nằm bên **phải**. Muốn biết `p` ở nhánh nào ta phải **duyệt** cây con để tìm.

In [3]:
root = build_tree([3, 5, 1, 6, 2, 0, 8, None, None, 7, 4])
print("gốc:", root.val, "| con trái:", root.left.val, "| con phải:", root.right.val)
print("5 > 3 nhưng nằm ở cây con TRÁI của 3:", root.left.val == 5)

gốc: 3 | con trái: 5 | con phải: 1
5 > 3 nhưng nằm ở cây con TRÁI của 3: True


### 3.3. Đệ quy trên cây: "tin vào lời gọi con"

Hàm đệ quy `f(node)` cho cây gốc `node`: ta giả định `f(node.left)` và `f(node.right)` đã cho đúng kết quả của hai cây con, rồi dùng chúng để tính kết quả của `node`. Trường hợp cơ sở: `node is None` thì trả `None`. Cách nghĩ này gọi là **hậu thứ tự** (xử lý con xong mới xử lý cha).

In [4]:
def count_nodes(node):
    if node is None:
        return 0
    return 1 + count_nodes(node.left) + count_nodes(node.right)


print(count_nodes(build_tree([3, 5, 1, 6, 2, 0, 8, None, None, 7, 4])))     # 9

9


### 3.4. Bản đồ cha (`parent`) và leo ngược lên gốc

Nút cây chỉ trỏ **xuống** con. Nếu duyệt cây một lần và ghi `parent[nút] = cha`, ta có thể **leo lên** từ bất kỳ nút nào về gốc, liệt kê mọi tổ tiên. Nút là khóa của `dict` được (hash theo danh tính đối tượng).

In [5]:
def build_parent_map(root):
    parent = {root: None}
    stack = [root]
    while stack:
        node = stack.pop()
        for child in (node.left, node.right):
            if child:
                parent[child] = node
                stack.append(child)
    return parent


root = build_tree([3, 5, 1, 6, 2, 0, 8, None, None, 7, 4])
parent = build_parent_map(root)
cur, chain = find_node(root, 4), []
while cur is not None:
    chain.append(cur.val)
    cur = parent[cur]
print("tổ tiên của 4 (từ chính nó lên gốc):", chain)      # [4, 2, 5, 3]

tổ tiên của 4 (từ chính nó lên gốc): [4, 2, 5, 3]


### 3.5. `set` các tổ tiên và leo tìm điểm gặp nhau

Nếu ta gom mọi tổ tiên của `p` vào một `set`, rồi leo lên từ `q` và dừng ở nút **đầu tiên** nằm trong `set` đó thì nút này chính là tổ tiên chung **thấp nhất** (vì ta leo từ thấp lên cao).

In [6]:
anc_p = set()
cur = find_node(root, 5)
while cur is not None:
    anc_p.add(cur)
    cur = parent[cur]
cur = find_node(root, 4)
while cur not in anc_p:
    cur = parent[cur]
print("LCA(5, 4) =", cur.val)                              # 5

LCA(5, 4) = 5


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Cây ví dụ 1:

```
            3
          /   \
         5     1
        / \   / \
       6   2 0   8
          / \
         7   4
```

**`p = 5`, `q = 1`:** `5` nằm trong cây con trái của `3`, `1` nằm trong cây con phải. Hai nút nằm hai phía của `3` nên nút thấp nhất chứa cả hai là `3`.

**`p = 5`, `q = 4`:** `4` nằm trong cây con của `5` (`5 → 2 → 4`). Nút `5` chứa cả hai (chính nó và `4`), và không nút nào thấp hơn `5` chứa `5`: LCA là `5`.

**`p = 7`, `q = 4`:** `7` và `4` là hai con của `2`: LCA là `2`.

**Cách nhìn đệ quy:** hỏi từng nút `v`: "trong cây con của tôi có `p`? có `q`?". Nút **thấp nhất** mà câu trả lời cho cả hai là "có" là LCA. Với mỗi nút, ta chỉ cần biết trong hai cây con bên dưới có tìm thấy `p`/`q` không.

In [7]:
root = build_tree([3, 5, 1, 6, 2, 0, 8, None, None, 7, 4])


def contains(node, target):
    stack = [node] if node else []
    while stack:
        cur = stack.pop()
        if cur is target:
            return True
        stack.extend(c for c in (cur.left, cur.right) if c)
    return False


p, q = find_node(root, 7), find_node(root, 4)
for v in (3, 5, 2, 7, 4):
    node = find_node(root, v)
    print(f"nút {v}: chứa p=7? {contains(node, p)}, chứa q=4? {contains(node, q)}")

nút 3: chứa p=7? True, chứa q=4? True
nút 5: chứa p=7? True, chứa q=4? True
nút 2: chứa p=7? True, chứa q=4? True
nút 7: chứa p=7? True, chứa q=4? False
nút 4: chứa p=7? False, chứa q=4? True


## 5. Cách 1 — Brute Force / baseline: kiểm tra mọi nút theo định nghĩa

**Ý tưởng.** Với **mỗi** nút `c` của cây: tính tập các nút trong cây con của `c`; nếu chứa cả `p` và `q` thì `c` là tổ tiên chung. Trong các tổ tiên chung chọn nút có **cây con nhỏ nhất** (nút thấp nhất). Đúng theo nghĩa đen định nghĩa nhưng chậm.

In [8]:
def lca_by_definition(root, p, q):
    def subtree_nodes(node):
        out, stack = set(), [node]
        while stack:
            cur = stack.pop()
            out.add(cur)
            if cur.left:
                stack.append(cur.left)
            if cur.right:
                stack.append(cur.right)
        return out

    best, best_size = None, None
    stack = [root]
    while stack:
        c = stack.pop()
        sub = subtree_nodes(c)                          # O(kích thước cây con của c)
        if p in sub and q in sub and (best is None or len(sub) < best_size):
            best, best_size = c, len(sub)
        if c.left:
            stack.append(c.left)
        if c.right:
            stack.append(c.right)
    return best


for values, pv, qv, expected in examples:
    root = build_tree(values)
    got = lca_by_definition(root, find_node(root, pv), find_node(root, qv))
    print(f"root={values}, p={pv}, q={qv} -> {got.val} (kỳ vọng {expected})")

root=[3, 5, 1, 6, 2, 0, 8, None, None, 7, 4], p=5, q=1 -> 3 (kỳ vọng 3)
root=[3, 5, 1, 6, 2, 0, 8, None, None, 7, 4], p=5, q=4 -> 5 (kỳ vọng 5)
root=[1, 2], p=1, q=2 -> 1 (kỳ vọng 1)


**Độ phức tạp (Cách 1).** Gọi `n` là số nút. Mỗi nút tính một tập con cây tốn tối đa `O(n)`: **Time `O(n²)`** (tệ nhất cây lệch), **Space `O(n)`**.

**Nút thắt.** Ta lặp lại việc duyệt cây con cho mọi nút. Cách tiết kiệm hơn: duyệt cây **một lần** để biết đường đi tới `p` và `q`, hoặc để biết cha của từng nút.

## 6. Ý tưởng thuật toán / bước đột phá

**Đệ quy hậu thứ tự trả về "nút tìm được".** Định nghĩa `f(v)` cho cây con gốc `v`:

- nếu `v` rỗng thì `f(v) = None`;
- nếu `v` là `p` hoặc `q` thì `f(v) = v` (dừng, không cần tìm sâu hơn: nếu nút còn lại nằm dưới `v` thì `v` là LCA, còn nếu không thì ta chỉ cần báo "tìm thấy một nút ở đây");
- ngược lại tính `left = f(v.left)`, `right = f(v.right)`:
  - nếu cả `left` và `right` **khác rỗng**: `p` và `q` nằm hai phía của `v`, nên `v` là LCA: trả `v`;
  - nếu chỉ một bên khác rỗng: trả bên đó (nó là nút `p`/`q` hoặc LCA đã tìm được ở dưới);
  - nếu cả hai rỗng: `None`.

Kết quả `f(root)` là LCA. Mỗi nút được thăm đúng một lần.

In [9]:
def f_trace(v, p, q):
    if v is None or v is p or v is q:
        return v
    left, right = f_trace(v.left, p, q), f_trace(v.right, p, q)
    return v if left and right else (left or right)


root = build_tree([3, 5, 1, 6, 2, 0, 8, None, None, 7, 4])
for pv, qv in [(5, 1), (5, 4), (7, 4), (6, 8)]:
    ans = f_trace(root, find_node(root, pv), find_node(root, qv))
    print(f"p={pv}, q={qv} -> LCA = {ans.val}")

p=5, q=1 -> LCA = 3
p=5, q=4 -> LCA = 5
p=7, q=4 -> LCA = 2
p=6, q=8 -> LCA = 3


## 7. Các cách tiếp theo

### Cách 2 — Đường đi gốc → nút (đệ quy quay lui) rồi tiền tố chung

**Giải thích.** Viết `find_path(v, target, path)`: thêm `v` vào `path`; nếu `v is target` thì xong; nếu tìm thấy ở con trái hoặc phải thì xong; ngược lại **`path.pop()`** (quay lui) rồi trả `False`. Với `path_p` và `path_q`, LCA là nút cuối của **tiền tố chung**.

In [10]:
def find_path(node, target, path):
    if node is None:
        return False
    path.append(node)
    if node is target:
        return True
    if find_path(node.left, target, path) or find_path(node.right, target, path):
        return True
    path.pop()                                   # quay lui: nút này không nằm trên đường đi
    return False


def lca_paths(root, p, q):
    path_p, path_q = [], []
    find_path(root, p, path_p)
    find_path(root, q, path_q)
    lca = None
    for a, b in zip(path_p, path_q):
        if a is not b:
            break
        lca = a
    return lca


for values, pv, qv, expected in examples:
    root = build_tree(values)
    got = lca_paths(root, find_node(root, pv), find_node(root, qv))
    print(f"root={values}, p={pv}, q={qv} -> {got.val} (kỳ vọng {expected})")

root=[3, 5, 1, 6, 2, 0, 8, None, None, 7, 4], p=5, q=1 -> 3 (kỳ vọng 3)
root=[3, 5, 1, 6, 2, 0, 8, None, None, 7, 4], p=5, q=4 -> 5 (kỳ vọng 5)
root=[1, 2], p=1, q=2 -> 1 (kỳ vọng 1)


**Độ phức tạp Cách 2:** **Time `O(n)`** (hai lần duyệt), **Space `O(h)`** cho đệ quy và đường đi (`h` là độ cao cây).

### Cách 3 — Bản đồ cha + tập tổ tiên (lặp)

**Giải thích.** Duyệt cây (ngăn xếp) ghi `parent` cho tới khi đã thấy cả `p` và `q` (dừng sớm). Gom mọi tổ tiên của `p` vào `set`, rồi leo từ `q` đến nút đầu tiên nằm trong `set`. Không dùng đệ quy nên an toàn với cây rất sâu.

In [11]:
def lca_parent_set(root, p, q):
    parent = {root: None}
    stack = [root]
    while p not in parent or q not in parent:         # dừng khi đã biết cha của cả p và q
        node = stack.pop()
        for child in (node.left, node.right):
            if child:
                parent[child] = node
                stack.append(child)
    ancestors = set()
    cur = p
    while cur is not None:
        ancestors.add(cur)                            # tổ tiên của p (kể cả chính p)
        cur = parent[cur]
    cur = q
    while cur not in ancestors:                       # leo từ q tới tổ tiên chung đầu tiên
        cur = parent[cur]
    return cur


for values, pv, qv, expected in examples:
    root = build_tree(values)
    got = lca_parent_set(root, find_node(root, pv), find_node(root, qv))
    print(f"root={values}, p={pv}, q={qv} -> {got.val} (kỳ vọng {expected})")

root=[3, 5, 1, 6, 2, 0, 8, None, None, 7, 4], p=5, q=1 -> 3 (kỳ vọng 3)
root=[3, 5, 1, 6, 2, 0, 8, None, None, 7, 4], p=5, q=4 -> 5 (kỳ vọng 5)
root=[1, 2], p=1, q=2 -> 1 (kỳ vọng 1)


**Độ phức tạp Cách 3:** **Time `O(n)`**, **Space `O(n)`** (bản đồ cha và tập tổ tiên).

### Cách 4 — Đệ quy hậu thứ tự (cách của lời giải tham khảo)

**Giải thích.** Chính là hàm `f` ở mục 6, viết gọn: nếu `root` rỗng, hoặc là `p`, hoặc là `q` thì trả `root`; đệ quy hai bên; nếu cả hai bên khác rỗng trả `root`, ngược lại trả bên khác rỗng.

In [12]:
def lca_recursive(root, p, q):
    if root is None or root is p or root is q:
        return root
    left = lca_recursive(root.left, p, q)
    right = lca_recursive(root.right, p, q)
    if left and right:
        return root                         # p và q ở hai phía của root
    return left or right                    # cả hai cùng một phía (hoặc chỉ tìm thấy một nút)


for values, pv, qv, expected in examples:
    root = build_tree(values)
    got = lca_recursive(root, find_node(root, pv), find_node(root, qv))
    print(f"root={values}, p={pv}, q={qv} -> {got.val} (kỳ vọng {expected})")

root=[3, 5, 1, 6, 2, 0, 8, None, None, 7, 4], p=5, q=1 -> 3 (kỳ vọng 3)
root=[3, 5, 1, 6, 2, 0, 8, None, None, 7, 4], p=5, q=4 -> 5 (kỳ vọng 5)
root=[1, 2], p=1, q=2 -> 1 (kỳ vọng 1)


In [13]:
# Code nộp LeetCode cho Cách 4
class Solution:
    def lowestCommonAncestor(self, root: 'TreeNode', p: 'TreeNode', q: 'TreeNode') -> 'TreeNode':
        if root in (None, p, q):
            return root
        left = self.lowestCommonAncestor(root.left, p, q)
        right = self.lowestCommonAncestor(root.right, p, q)
        return root if left and right else (left or right)

**Độ phức tạp Cách 4:** **Time `O(n)`** (mỗi nút thăm một lần), **Space `O(h)`** ngăn xếp đệ quy; cây lệch `h = n` có thể gây `RecursionError` (xem phần test).

### Cách 5 — Bản đồ cha + độ sâu (hai nút "leo" đến khi gặp nhau)

**Giải thích.** Duyệt cây ghi `parent` và `depth`. Đưa nút **sâu hơn** lên cho bằng độ sâu của nút kia, rồi leo **đồng thời** cả hai lên đến khi trùng nhau. Giống bài "giao điểm hai danh sách liên kết". Không dùng `set`, và không đệ quy.

In [14]:
def lca_depth_climb(root, p, q):
    parent = {root: None}
    depth = {root: 0}
    stack = [root]
    while p not in parent or q not in parent:
        node = stack.pop()
        for child in (node.left, node.right):
            if child:
                parent[child] = node
                depth[child] = depth[node] + 1
                stack.append(child)
    a, b = p, q
    while depth[a] > depth[b]:                        # đưa nút sâu hơn lên cùng độ sâu
        a = parent[a]
    while depth[b] > depth[a]:
        b = parent[b]
    while a is not b:                                 # leo cùng lúc tới khi gặp nhau
        a, b = parent[a], parent[b]
    return a


for values, pv, qv, expected in examples:
    root = build_tree(values)
    got = lca_depth_climb(root, find_node(root, pv), find_node(root, qv))
    print(f"root={values}, p={pv}, q={qv} -> {got.val} (kỳ vọng {expected})")

root=[3, 5, 1, 6, 2, 0, 8, None, None, 7, 4], p=5, q=1 -> 3 (kỳ vọng 3)
root=[3, 5, 1, 6, 2, 0, 8, None, None, 7, 4], p=5, q=4 -> 5 (kỳ vọng 5)
root=[1, 2], p=1, q=2 -> 1 (kỳ vọng 1)


**Độ phức tạp Cách 5:** **Time `O(n)`**, **Space `O(n)`** (hai bản đồ).

> Cách 4 ngắn và là cách nộp bài chuẩn. Cách 3 và 5 là lựa chọn lặp khi lo `RecursionError`. Cách 1 là chuẩn theo định nghĩa.

## 8. TRACE từng bước

Ta theo dõi Cách 4 (đệ quy) trên `p = 5`, `q = 1` (hai phía) và `p = 5`, `q = 4` (một nút là tổ tiên của nút kia), in vào/ra từng nút với thụt dòng theo độ sâu.

In [15]:
def trace_recursive(root, p, q, depth=0):
    pad = "  " * depth
    if root is None:
        print(f"{pad}None -> trả None")
        return None
    if root is p or root is q:
        print(f"{pad}nút {root.val} là {'p' if root is p else 'q'} -> trả chính nó")
        return root
    print(f"{pad}vào nút {root.val}")
    left = trace_recursive(root.left, p, q, depth + 1)
    right = trace_recursive(root.right, p, q, depth + 1)
    lv = left.val if left else None
    rv = right.val if right else None
    if left and right:
        print(f"{pad}nút {root.val}: left={lv}, right={rv} -> cả hai khác rỗng: LCA = {root.val}")
        return root
    print(f"{pad}nút {root.val}: left={lv}, right={rv} -> trả {'left' if left else 'right'}")
    return left or right


tree = build_tree([3, 5, 1, 6, 2, 0, 8, None, None, 7, 4])
assert trace_recursive(tree, find_node(tree, 5), find_node(tree, 1)).val == 3
print()
assert trace_recursive(tree, find_node(tree, 5), find_node(tree, 4)).val == 5

vào nút 3
  nút 5 là p -> trả chính nó
  nút 1 là q -> trả chính nó
nút 3: left=5, right=1 -> cả hai khác rỗng: LCA = 3

vào nút 3
  nút 5 là p -> trả chính nó
  vào nút 1
    vào nút 0
      None -> trả None
      None -> trả None
    nút 0: left=None, right=None -> trả right
    vào nút 8
      None -> trả None
      None -> trả None
    nút 8: left=None, right=None -> trả right
  nút 1: left=None, right=None -> trả right
nút 3: left=5, right=None -> trả left


Bảng tóm tắt (`p = 5`, `q = 1`):

| nút | `left` | `right` | kết quả |
|---|---|---|---|
| `5` | (là `p`: trả chính nó) | | `5` |
| `1` | (là `q`: trả chính nó) | | `1` |
| `3` | `5` | `1` | cả hai khác rỗng: **trả `3`** |

Với `p = 5`, `q = 4`: nút `5` là `p` nên dừng và trả `5` ngay (không cần đi sâu vào cây con của `5`); nút `1` cho `None`; ở `3`: `left = 5`, `right = None` nên trả `5`. Đúng với kỳ vọng (`5`).

## 9. Code nộp LeetCode

Bản nộp dùng Cách 4 (đệ quy hậu thứ tự). Đúng class `Solution`, đúng method `lowestCommonAncestor(self, root, p, q)`.

In [16]:
class Solution:
    def lowestCommonAncestor(self, root: 'TreeNode', p: 'TreeNode', q: 'TreeNode') -> 'TreeNode':
        if root is None or root is p or root is q:     # cơ sở: rỗng, hoặc gặp p / q thì trả luôn
            return root
        left = self.lowestCommonAncestor(root.left, p, q)
        right = self.lowestCommonAncestor(root.right, p, q)
        if left and right:                             # p, q ở hai phía: root là LCA
            return root
        return left or right                           # cùng một phía: trả phía đã tìm thấy


for values, pv, qv, expected in examples:
    root = build_tree(values)
    got = Solution().lowestCommonAncestor(root, find_node(root, pv), find_node(root, qv))
    assert got.val == expected and got is find_node(root, expected)
print("PASS ví dụ đề cho class Solution")

PASS ví dụ đề cho class Solution


## 10. Vì sao thuật toán đúng

Xét `f(v)` như ở mục 6, với giả thiết `p` và `q` đều có trong cây và `p ≠ q`. Ta chứng minh bằng quy nạp: **`f(v)`** là
- `None` nếu cây con của `v` không chứa `p` lẫn `q`;
- `p` hoặc `q` nếu cây con chứa **đúng một** trong hai;
- LCA của `p` và `q` nếu cây con chứa cả hai.

Cơ sở: `v` rỗng cho `None`; `v` là `p` hoặc `q` thì trả `v`: đúng ở các trường hợp "chứa đúng một" và "chứa cả hai" (khi nút kia nằm dưới `v` thì `v` là LCA; ta không cần đi xuống thêm).

Bước quy nạp với `v` không phải `p`, `q`:

- cả hai con cho kết quả khác rỗng: mỗi cây con chứa ít nhất một trong hai nút; vì chỉ có hai nút nên mỗi cây con chứa đúng một nút, `p` và `q` ở hai phía: `v` là tổ tiên chung, và không nút nào dưới `v` chứa cả hai, nên `v` là LCA;
- đúng một con khác rỗng: nếu đó là `p` hoặc `q` thì cây con chứa đúng một nút (nút kia không nằm trong cây con của `v`), trả nút đó là đúng; nếu đó là LCA thì cây con của `v` chứa cả hai và LCA nằm dưới, truyền lên nguyên vẹn;
- cả hai rỗng: không chứa gì.

Ở gốc, cây chứa cả hai nên `f(root)` là LCA.

Các cách khác:

- **Cách 1:** đúng theo định nghĩa (tổ tiên chung có cây con nhỏ nhất, vì các tổ tiên chung xếp thành một chuỗi lồng nhau).
- **Cách 2:** mọi tổ tiên của một nút nằm trên đường đi gốc → nút; tổ tiên chung là tiền tố chung; LCA là nút cuối tiền tố.
- **Cách 3 và 5:** leo cha từ `q` (hoặc cả hai nút cùng độ sâu) gặp nút đầu tiên trên đường đi gốc của `p`: đó là tổ tiên chung sâu nhất.

## 11. Phân tích Time / Space Complexity

Gọi `n` là số nút, `h` là độ cao cây (`log n ≤ h ≤ n`).

- **Cách 1 (theo định nghĩa):** Time `O(n²)`, Space `O(n)`.
- **Cách 2 (đường đi + tiền tố chung):** Time `O(n)`, Space `O(h)`.
- **Cách 3 (bản đồ cha + tập tổ tiên):** Time `O(n)`, Space `O(n)`.
- **Cách 4 (đệ quy hậu thứ tự):** Time `O(n)`, Space `O(h)`.
- **Cách 5 (bản đồ cha + độ sâu):** Time `O(n)`, Space `O(n)`.
- Trường hợp xấu nhất: cây lệch có `h = n`, đệ quy (Cách 2, 4) dễ vượt giới hạn đệ quy của Python (~1000) trong khi Cách 3 và 5 dùng vòng lặp nên an toàn.

## 12. Bộ test

Ta gom các cách vào từ điển `impls`. `oracle` đi **đúng theo định nghĩa** (nút có cây con nhỏ nhất chứa cả hai, tính độc lập bằng tập các nút). Cây ngẫu nhiên: gắn lần lượt từng nút mới vào một vị trí trống ngẫu nhiên của cây hiện có, nên hình dạng rất đa dạng (không phải BST).

In [17]:
sol = Solution()
impls = {
    "by_definition": lca_by_definition,
    "paths": lca_paths,
    "parent_set": lca_parent_set,
    "recursive": lca_recursive,
    "depth_climb": lca_depth_climb,
    "submit": sol.lowestCommonAncestor,
}


def oracle(root, p, q):
    return lca_by_definition(root, p, q)


def random_tree(n):
    """Cây nhị phân ngẫu nhiên n nút (giá trị phân biệt), gắn mỗi nút mới vào một chỗ trống ngẫu nhiên."""
    values = random.sample(range(-1000, 1001), n)
    root = TreeNode(values[0])
    free = [(root, "left"), (root, "right")]
    for v in values[1:]:
        i = random.randrange(len(free))
        free[i], free[-1] = free[-1], free[i]
        parent, side = free.pop()
        node = TreeNode(v)
        setattr(parent, side, node)
        free.append((node, "left"))
        free.append((node, "right"))
    return root, values


for values, pv, qv, expected in examples:
    root = build_tree(values)
    p, q = find_node(root, pv), find_node(root, qv)
    assert oracle(root, p, q).val == expected
    for name, fn in impls.items():
        got = fn(root, p, q)
        assert got is oracle(root, p, q), (name, values, pv, qv, got.val)
print("PASS ví dụ của đề (mọi cách và oracle độc lập, so sánh theo danh tính nút)")

PASS ví dụ của đề (mọi cách và oracle độc lập, so sánh theo danh tính nút)


### 12.1. Các trường hợp biên

In [18]:
def check_all(root, p, q, expected_val, label, skip=()):
    for name, fn in impls.items():
        if name in skip:
            continue
        got = fn(root, p, q)
        assert got.val == expected_val, (label, name, got.val, expected_val)
    print("PASS", label, "->", expected_val)


r = build_tree([1, 2])
check_all(r, find_node(r, 1), find_node(r, 2), 1, "hai nút: gốc là tổ tiên")
check_all(r, find_node(r, 2), find_node(r, 1), 1, "đảo thứ tự p, q")
r = build_tree([3, 5, 1, 6, 2, 0, 8, None, None, 7, 4])
check_all(r, find_node(r, 6), find_node(r, 8), 3, "hai lá ở hai nhánh lớn khác nhau")
check_all(r, find_node(r, 7), find_node(r, 4), 2, "hai lá cùng cha")
check_all(r, find_node(r, 6), find_node(r, 4), 5, "hai nút trong cây con của 5")
check_all(r, find_node(r, 3), find_node(r, 8), 3, "p là gốc")
check_all(r, find_node(r, 5), find_node(r, 7), 5, "q nằm sâu dưới p")
r = build_tree([1, None, 2, None, 3, None, 4])
check_all(r, find_node(r, 3), find_node(r, 4), 3, "cây lệch phải nhỏ")
r = build_tree([-10**9, 10**9])
check_all(r, find_node(r, -10**9), find_node(r, 10**9), -10**9, "giá trị biên +-10^9")

# Cây lệch (zig-zag) sâu 3000: bỏ qua các cách đệ quy và cách O(n^2)
root = TreeNode(0)
cur = root
for v in range(1, 3000):
    node = TreeNode(v)
    if v % 2:
        cur.left = node
    else:
        cur.right = node
    cur = node
deep = find_node(root, 2999)
check_all(root, find_node(root, 5), deep, 5, "cây zig-zag sâu 3000", skip=("by_definition", "paths", "recursive", "submit"))
check_all(root, find_node(root, 2998), deep, 2998, "cây zig-zag: hai nút cuối", skip=("by_definition", "paths", "recursive", "submit"))
try:
    lca_recursive(root, find_node(root, 5), deep)
    print("(đệ quy không gặp lỗi ở giới hạn hiện tại)")
except RecursionError:
    print("lca_recursive: RecursionError trên cây sâu 3000 (Cách 3 và 5 dùng vòng lặp nên không bị)")

PASS hai nút: gốc là tổ tiên -> 1
PASS đảo thứ tự p, q -> 1
PASS hai lá ở hai nhánh lớn khác nhau -> 3
PASS hai lá cùng cha -> 2
PASS hai nút trong cây con của 5 -> 5
PASS p là gốc -> 3
PASS q nằm sâu dưới p -> 5
PASS cây lệch phải nhỏ -> 3
PASS giá trị biên +-10^9 -> -1000000000
PASS cây zig-zag sâu 3000 -> 5
PASS cây zig-zag: hai nút cuối -> 2998
(đệ quy không gặp lỗi ở giới hạn hiện tại)


### 12.2. Tính chất và phản ví dụ

In [19]:
random.seed(236)

# Tính chất 1: đối xứng; LCA(p, tổ tiên của p) = tổ tiên đó
for _ in range(300):
    n = random.randint(2, 40)
    root, values = random_tree(n)
    pv, qv = random.sample(values, 2)
    p, q = find_node(root, pv), find_node(root, qv)
    assert sol.lowestCommonAncestor(root, p, q) is sol.lowestCommonAncestor(root, q, p)
print("PASS tính chất: đối xứng")

for _ in range(200):
    n = random.randint(2, 40)
    root, values = random_tree(n)
    parent = build_parent_map(root)
    node = find_node(root, random.choice(values))
    anc = node
    for _ in range(random.randint(0, 5)):
        if parent[anc] is not None:
            anc = parent[anc]
    if anc is not node:
        assert sol.lowestCommonAncestor(root, node, anc) is anc
print("PASS tính chất: LCA(nút, tổ tiên của nút) là chính tổ tiên đó")

# Phản ví dụ: dùng thuật toán của BST (so sánh giá trị) trên cây thường
def sai_dung_bst(root, p, q):
    lo, hi = min(p.val, q.val), max(p.val, q.val)
    steps = 0
    while root and steps < 100:
        if root.val < lo:
            root = root.right
        elif root.val > hi:
            root = root.left
        else:
            return root
        steps += 1
    return None

r = build_tree([3, 5, 1, 6, 2, 0, 8, None, None, 7, 4])
got = sai_dung_bst(r, find_node(r, 6), find_node(r, 4))
print("p=6, q=4 -> sai (thuật toán BST):", got.val if got else None, "| đúng:", sol.lowestCommonAncestor(r, find_node(r, 6), find_node(r, 4)).val)
assert got is None or got.val != 5

PASS tính chất: đối xứng
PASS tính chất: LCA(nút, tổ tiên của nút) là chính tổ tiên đó
p=6, q=4 -> sai (thuật toán BST): None | đúng: 5


### 12.3. Đối chiếu ngẫu nhiên (có seed) với oracle độc lập

In [20]:
random.seed(2026)
for trial in range(1500):
    n = random.randint(2, 40)
    root, values = random_tree(n)
    pv, qv = random.sample(values, 2)
    p, q = find_node(root, pv), find_node(root, qv)
    expected = oracle(root, p, q)
    for name, fn in impls.items():
        got = fn(root, p, q)
        assert got is expected, (name, pv, qv, got.val, expected.val)

for trial in range(30):                           # cây lớn: các cách O(n) đối chiếu lẫn nhau
    n = random.randint(500, 1500)
    root, values = random_tree(n)
    pv, qv = random.sample(values, 2)
    p, q = find_node(root, pv), find_node(root, qv)
    a = lca_parent_set(root, p, q)
    assert a is lca_depth_climb(root, p, q) is lca_paths(root, p, q) is lca_recursive(root, p, q)
print("PASS 1530 cây ngẫu nhiên: mọi cách khớp oracle theo định nghĩa (so sánh danh tính nút)")

PASS 1530 cây ngẫu nhiên: mọi cách khớp oracle theo định nghĩa (so sánh danh tính nút)


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Dùng thuật toán của BST cho cây nhị phân thường

In [21]:
r = build_tree([3, 5, 1, 6, 2, 0, 8, None, None, 7, 4])
got = sai_dung_bst(r, find_node(r, 7), find_node(r, 4))
print("p=7, q=4 -> sai:", got.val if got else None, "| đúng:", Solution().lowestCommonAncestor(r, find_node(r, 7), find_node(r, 4)).val)

p=7, q=4 -> sai: None | đúng: 2


**Vì sao sai:** trong cây thường **không có thứ tự giá trị** (`5` nằm bên trái của `3` dù `5 > 3`). So sánh `root.val` với `p.val`, `q.val` không cho biết nên đi nhánh nào. Phải duyệt cây để tìm `p`, `q`.

### Lỗi 2: Điều kiện cơ sở thiếu `root is q` (hoặc `root is p`)

In [22]:
def sai_thieu_co_so(root, p, q):
    if root is None or root is p:                     # sai: thiếu `or root is q`
        return root
    left = sai_thieu_co_so(root.left, p, q)
    right = sai_thieu_co_so(root.right, p, q)
    if left and right:
        return root
    return left or right


r = build_tree([3, 5, 1, 6, 2, 0, 8, None, None, 7, 4])
got = sai_thieu_co_so(r, find_node(r, 5), find_node(r, 1))
print("p=5, q=1 -> sai:", got.val if got else None, "| đúng:", Solution().lowestCommonAncestor(r, find_node(r, 5), find_node(r, 1)).val)

p=5, q=1 -> sai: 5 | đúng: 3


**Vì sao sai:** nút `1` (là `q`) không được nhận ra nên đệ quy đi tiếp xuống con của `1` và trả `None`; ở gốc `left = 5`, `right = None` nên trả `5` thay vì `3`. Cần kiểm tra **cả hai**: `root is p or root is q`.

### Lỗi 3: Chỉ trả `left or right`, quên trường hợp cả hai bên đều tìm thấy

In [23]:
def sai_quen_ca_hai(root, p, q):
    if root is None or root is p or root is q:
        return root
    left = sai_quen_ca_hai(root.left, p, q)
    right = sai_quen_ca_hai(root.right, p, q)
    return left or right                              # sai: khi cả hai khác rỗng phải trả root


r = build_tree([3, 5, 1, 6, 2, 0, 8, None, None, 7, 4])
got = sai_quen_ca_hai(r, find_node(r, 5), find_node(r, 1))
print("p=5, q=1 -> sai:", got.val, "| đúng:", Solution().lowestCommonAncestor(r, find_node(r, 5), find_node(r, 1)).val)

p=5, q=1 -> sai: 5 | đúng: 3


**Vì sao sai:** khi `left` và `right` đều khác rỗng, `p` và `q` nằm ở hai phía của nút hiện tại nên nút này là LCA. `left or right` luôn chọn `left` và bỏ mất thông tin về bên phải, trả `5` thay vì `3`.

### Lỗi 4: Quên `path.pop()` khi quay lui trong hàm tìm đường đi

In [24]:
def sai_find_path(node, target, path):
    if node is None:
        return False
    path.append(node)
    if node is target:
        return True
    if sai_find_path(node.left, target, path) or sai_find_path(node.right, target, path):
        return True
    return False                                      # sai: thiếu path.pop()


r = build_tree([3, 5, 1, 6, 2, 0, 8, None, None, 7, 4])
path = []
sai_find_path(r, find_node(r, 4), path)
print("đường đi sai tới 4:", [n.val for n in path])
good = []
find_path(r, find_node(r, 4), good)
print("đường đi đúng tới 4:", [n.val for n in good])

đường đi sai tới 4: [3, 5, 6, 2, 7, 4]
đường đi đúng tới 4: [3, 5, 2, 4]


**Vì sao sai:** khi một nhánh không chứa `target`, ta phải **gỡ** nút khỏi `path` trước khi thử nhánh khác. Không `pop` thì `path` chứa cả các nút đã thử nhưng sai (`6`, `7`...), nên tiền tố chung của hai đường đi bị tính sai.

## 14. Mẹo làm nhanh & nhận dạng pattern

- LCA trong cây **nhị phân thường** → **đệ quy hậu thứ tự** `left`, `right`: cả hai khác rỗng → trả nút hiện tại; ngược lại trả bên khác rỗng.
- Nếu cây là **BST**, dùng bài 235 (so sánh giá trị, `O(h)`) thay vì duyệt hết cây.
- Cần LCA **nhiều lần** trên cùng một cây: dựng bản đồ cha + độ sâu (hoặc binary lifting / Euler tour) một lần.
- Lo `RecursionError`: dùng bản đồ cha + vòng lặp (Cách 3, 5).
- Khi quay lui ghi đường đi: nhớ `pop`.
- Test cây: `build_tree` + `find_node` + cây ngẫu nhiên + oracle theo định nghĩa "tổ tiên chung có cây con nhỏ nhất" là bộ khung rất mạnh.

## 15. Biến thể của bài

1. **LCA trong BST** (bài 235).
2. **`p` hoặc `q` có thể không có trong cây** (LeetCode 1644): phải đếm số nút tìm thấy.
3. **LCA của nhiều nút** (LeetCode 1676).
4. **Khoảng cách giữa hai nút** = `depth(p) + depth(q) − 2·depth(LCA)`.
5. **LCA của các nút sâu nhất** (LeetCode 865).

In [25]:
def lca_maybe_absent(root, p, q):
    """Biến thể 2: trả None nếu p hoặc q không có trong cây. Đệ quy đếm số nút tìm thấy."""
    found = 0
    answer = None

    def dfs(node):
        nonlocal found, answer
        if node is None:
            return 0
        hit_left = dfs(node.left)
        hit_right = dfs(node.right)
        here = 1 if node is p or node is q else 0
        total = hit_left + hit_right + here
        if total == 2 and answer is None:
            answer = node                      # nút đầu tiên (thấp nhất) mà cây con chứa cả hai
        return total

    dfs(root)
    return answer


def lca_of_nodes(root, nodes):
    """Biến thể 3: LCA của một tập nút bằng đệ quy."""
    targets = set(nodes)

    def dfs(node):
        if node is None or node in targets:
            return node
        left, right = dfs(node.left), dfs(node.right)
        return node if left and right else (left or right)

    return dfs(root)


def distance_between(root, p, q):
    """Biến thể 4: số cạnh giữa p và q."""
    parent = build_parent_map(root)

    def depth(x):
        d = 0
        while parent[x] is not None:
            x = parent[x]
            d += 1
        return d

    lca = lca_depth_climb(root, p, q)
    return depth(p) + depth(q) - 2 * depth(lca)


r = build_tree([3, 5, 1, 6, 2, 0, 8, None, None, 7, 4])
ghost = TreeNode(99)
assert lca_maybe_absent(r, find_node(r, 5), find_node(r, 1)) is r
assert lca_maybe_absent(r, find_node(r, 5), ghost) is None
assert lca_of_nodes(r, [find_node(r, 7), find_node(r, 4), find_node(r, 6)]).val == 5
assert distance_between(r, find_node(r, 7), find_node(r, 8)) == 5
random.seed(15)
for _ in range(200):
    root, values = random_tree(random.randint(2, 30))
    a, b = random.sample(values, 2)
    p, q = find_node(root, a), find_node(root, b)
    assert lca_maybe_absent(root, p, q) is oracle(root, p, q)
print("PASS biến thể: nút vắng mặt, nhiều nút, khoảng cách")

PASS biến thể: nút vắng mặt, nhiều nút, khoảng cách


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Theo định nghĩa | tập cây con của từng nút, chọn nhỏ nhất | `O(n²)` | `O(n)` | oracle, hiểu đề |
| 2. Đường đi gốc → nút | quay lui + tiền tố chung | `O(n)` | `O(h)` | dễ hiểu từ khái niệm tổ tiên |
| 3. Bản đồ cha + tập tổ tiên | leo từ `q` đến nút trong tập của `p` | `O(n)` | `O(n)` | lặp, không lo đệ quy sâu |
| 4. Đệ quy hậu thứ tự | trả nút tìm được, hai bên khác rỗng → LCA | `O(n)` | `O(h)` | **cách nộp bài** |
| 5. Bản đồ cha + độ sâu | cân độ sâu rồi leo cùng lúc | `O(n)` | `O(n)` | lặp, không dùng `set` |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `count_nodes(build_tree([1, 2, 3]))`, `Solution().lowestCommonAncestor` (giá trị) của `2` và `3` trong cây `[1, 2, 3]`, của `1` và `3`, của `2` và `2` nếu cho phép `p = q`, và nếu `root is p` thì hàm trả về gì ngay lập tức?

**Bài 2 (tính tay).** Cây `[3,5,1,6,2,0,8,null,null,7,4]`: tìm LCA của `(6, 2)`, `(7, 8)`, `(0, 8)`, `(6, 4)`. Vẽ cây con của mỗi tổ tiên để kiểm tra "chứa cả hai".

**Bài 3 (điền TODO).** Hoàn thành `lca_todo(root, p, q)` theo Cách 4 (đệ quy).

In [26]:
def lca_todo(root, p, q):
    # TODO 1: nếu root rỗng, hoặc là p, hoặc là q thì trả root
    # TODO 2: left = lca_todo(root.left, p, q); right = lca_todo(root.right, p, q)
    # TODO 3: nếu cả left và right khác rỗng trả root; ngược lại trả left or right
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (các TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `lca_depth2(root, p, q)` theo Cách 5 (bản đồ cha và độ sâu, cân độ sâu rồi leo). Kiểm tra bằng ba ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** (a) Viết `path_between(root, p, q)` trả về danh sách **giá trị** các nút trên đường đi từ `p` lên LCA rồi xuống `q`. (b) Viết `is_ancestor(root, a, b)` trả `True` nếu nút `a` là tổ tiên (có thể là chính nó) của nút `b`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: cây `[1,2,3]` có `1` là gốc với hai con `2` và `3`; nếu gốc là `p` thì điều kiện cơ sở trả ngay gốc. Bài 2: hỏi "nút nào thấp nhất chứa cả hai".
- **Mức 2 (vừa):** Bài 1: `3`; `1`; `1`; `2` (cho phép `p = q`); trả chính `root`. Bài 2: `(6,2)`: `5`; `(7,8)`: `3`; `(0,8)`: `1`; `(6,4)`: `5`. Bài 3: ba TODO trong comment.
- **Mức 3 (sâu):** Bài 4: `parent` và `depth` bằng ngăn xếp; trong khi `depth[a] > depth[b]`: `a = parent[a]`, rồi leo cả hai đến khi `a is b`. Bài 5(a): đường từ `p` lên `lca` (theo `parent`), rồi đoạn từ `lca` xuống `q` (đảo ngược đường từ `q` lên). Bài 5(b): `a` là tổ tiên của `b` khi `LCA(a, b) is a` (hoặc leo từ `b`).

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [27]:
# Bài 1
r1 = build_tree([1, 2, 3])
assert count_nodes(r1) == 3
assert Solution().lowestCommonAncestor(r1, find_node(r1, 2), find_node(r1, 3)).val == 1
assert Solution().lowestCommonAncestor(r1, find_node(r1, 1), find_node(r1, 3)).val == 1
assert Solution().lowestCommonAncestor(r1, find_node(r1, 2), find_node(r1, 2)).val == 2
assert Solution().lowestCommonAncestor(r1, r1, find_node(r1, 3)) is r1

# Bài 2
r2 = build_tree([3, 5, 1, 6, 2, 0, 8, None, None, 7, 4])
for pv, qv, expected in [(6, 2, 5), (7, 8, 3), (0, 8, 1), (6, 4, 5)]:
    assert Solution().lowestCommonAncestor(r2, find_node(r2, pv), find_node(r2, qv)).val == expected

# Bài 3
def lca_todo_ans(root, p, q):
    if root is None or root is p or root is q:
        return root
    left = lca_todo_ans(root.left, p, q)
    right = lca_todo_ans(root.right, p, q)
    if left and right:
        return root
    return left or right

for values, pv, qv, expected in examples:
    rt = build_tree(values)
    assert lca_todo_ans(rt, find_node(rt, pv), find_node(rt, qv)).val == expected

# Bài 4
def lca_depth2(root, p, q):
    parent = {root: None}
    depth = {root: 0}
    stack = [root]
    while stack:
        node = stack.pop()
        for child in (node.left, node.right):
            if child:
                parent[child] = node
                depth[child] = depth[node] + 1
                stack.append(child)
    a, b = p, q
    while depth[a] > depth[b]:
        a = parent[a]
    while depth[b] > depth[a]:
        b = parent[b]
    while a is not b:
        a, b = parent[a], parent[b]
    return a

for values, pv, qv, expected in examples:
    rt = build_tree(values)
    assert lca_depth2(rt, find_node(rt, pv), find_node(rt, qv)).val == expected

# Bài 5(a), 5(b)
def path_between(root, p, q):
    parent = build_parent_map(root)
    lca = lca_depth2(root, p, q)
    up, cur = [], p
    while cur is not lca:
        up.append(cur.val)
        cur = parent[cur]
    up.append(lca.val)
    down, cur = [], q
    while cur is not lca:
        down.append(cur.val)
        cur = parent[cur]
    return up + down[::-1]

def is_ancestor(root, a, b):
    return lca_depth2(root, a, b) is a if a is not b else True

assert path_between(r2, find_node(r2, 7), find_node(r2, 8)) == [7, 2, 5, 3, 1, 8]
assert path_between(r2, find_node(r2, 5), find_node(r2, 4)) == [5, 2, 4]
assert is_ancestor(r2, find_node(r2, 5), find_node(r2, 4)) and not is_ancestor(r2, find_node(r2, 1), find_node(r2, 4))
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (LCA trong cây nhị phân bằng đệ quy hậu thứ tự)

```python
class Solution:
    def lowestCommonAncestor(self, root: 'TreeNode', p: 'TreeNode', q: 'TreeNode') -> 'TreeNode':
        if root is None or root is p or root is q:
            return root
        left = self.lowestCommonAncestor(root.left, p, q)
        right = self.lowestCommonAncestor(root.right, p, q)
        if left and right:
            return root
        return left or right
```

**Checklist**

- [ ] Điều kiện cơ sở có **cả ba**: rỗng, `root is p`, `root is q`?
- [ ] Khi `left` và `right` đều khác rỗng trả `root`; ngược lại trả bên khác rỗng (`left or right`)?
- [ ] Không dùng so sánh giá trị kiểu BST trên cây thường?
- [ ] Hiểu "nút là hậu duệ của chính nó": gặp `p` thì dừng, trả luôn?
- [ ] Cân nhắc cây sâu (`n` tới `10⁵`): đệ quy có thể vượt giới hạn, có thể chuyển sang bản đồ cha + vòng lặp?
- [ ] Đã thử: hai nút, `p` là tổ tiên của `q`, hai lá cùng cha, `p`/`q` đảo thứ tự?
- [ ] Đã đối chiếu với oracle theo định nghĩa trên cây ngẫu nhiên?
- [ ] Skeleton đúng `class Solution` / `def lowestCommonAncestor(self, root: 'TreeNode', p: 'TreeNode', q: 'TreeNode') -> 'TreeNode'`?